# SurEau photosynthesis
 

In [ ]:
#| default_exp sureau_photosynthesis

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export

import numpy as np
from plant_hydraulics.utils import arrhenius_function, inhibition_function   
from plant_hydraulics.parameter_classes import (
    PhysCon,                                                                 
    SurEauVegetationParams,
    SurEauPlantFluxes,
)

In [ ]:
#| export

def _quadp(a, b, c):
    """Larger root of a·x² + b·x + c = 0 (similar to plantecophys ).
    Returns 0 for imaginary roots or a == 0 with b == 0."""
    disc = b * b - 4.0 * a * c
    if disc < 0.0:
        return 0.0
    if a == 0.0:
        return 0.0 if b == 0.0 else -c / b
    return (-b + np.sqrt(disc)) / (2.0 * a)


In [ ]:
#| export

def _getci(VJ, GSDIVA, PAR, Ca, g0c, Rd, Vcmax, Jmax, Km, Gstar):
    """Coupled operating Ci for the Rubisco- and RuBP-limited rates.

    Exact port of plantecophys `getCI` (photosyn.R): substitutes the Medlyn
    supply (gs = g0 + GSDIVA·A, CO₂ basis) into each Farquhar demand and solves
    the resulting quadratic in Ci with the larger root. `g0c` and `GSDIVA` are on
    a CO₂ basis. Returns (CIJ, CIC).
    """
    if PAR == 0.0 or VJ == 0.0:
        return Ca, Ca
    
    # Rubisco-limited -----------------------------------------------------------
    A = g0c + GSDIVA * (Vcmax - Rd)
    
    B = (1.0 - Ca * GSDIVA) * (Vcmax - Rd) + g0c * (Km - Ca) \
        - GSDIVA * (Vcmax * Gstar + Km * Rd)
    
    C = -(1.0 - Ca * GSDIVA) * (Vcmax * Gstar + Km * Rd) - g0c * Km * Ca
    
    CIC = _quadp(A, B, C)
    
    # RuBP-regeneration-limited -------------------------------------------------
    A = g0c + GSDIVA * (VJ - Rd)
    
    B = (1.0 - Ca * GSDIVA) * (VJ - Rd) + g0c * (2.0 * Gstar - Ca) \
        - GSDIVA * (VJ * Gstar + 2.0 * Gstar * Rd)
    
    C = -(1.0 - Ca * GSDIVA) * Gstar * (VJ + 2.0 * Rd) - g0c * 2.0 * Gstar * Ca
    
    CIJ = _quadp(A, B, C)
    
    
    return CIJ, CIC

In [ ]:
#| export
def solve_coupled_medlyn_fvcb(T_leaf, PAR, VPD, params):
    """Medlyn-coupled FvCB operating point reproduces plantecophys `Photosyn`,
    except Rd uses a peaked Arrhenius (plantecophys uses Q10).

    Solves the gs–A–Ci system analytically in one pass (no iteration): the Medlyn
    slope is baked into `_getci`'s quadratic,as in plantecophys. There is
    no CO₂ boundary layer (cs = Ca), matching the basic `Photosyn` path.


    Parameters
    ----------
    T_leaf : float   Leaf temperature [°C].
    PAR    : float   Incident PAR [µmol m⁻² s⁻¹] (absorptance folded into alpha_j).
    VPD    : float   Leaf-to-air VPD [kPa]; floored at params.vpdmin.
    params : SurEauVegetationParams

    Returns
    -------
    An     : Net photosynthesis [µmol CO₂ m⁻² s⁻¹] (= −Rd at night).
    ci     : Intercellular CO₂ [µmol mol⁻¹].
    cs     : Leaf-surface CO₂ [µmol mol⁻¹] (= Ca; no boundary layer).
    gs_mol : Stomatal conductance to H₂O [mol m⁻² s⁻¹].
    """
    # Normalization function ----------------------------------------------------
    def _fth25(hd, se):
        return 1.0 + np.exp((se * T0 - hd) / (PhysCon.rgas * T0))
    

    # Constants -----------------------------------------------------------------
    # Kelvin (utils expect K)
    tl = T_leaf + PhysCon.tfrz
    
    # plantecophys Patm correction                       
    pcor = params.Patm_photo / 100.0                 

    # Michaelis–Menten and Γ* (monotonic Arrhenius)
    Kc = params.kc25 * arrhenius_function(tl, params.kcha)
    Ko = params.ko25 * arrhenius_function(tl, params.koha)
    Gstar = params.cp25 * arrhenius_function(tl, params.cpha) * pcor
    Oi = params.O2_air * pcor
    Km = Kc * (1.0 + Oi / Ko)

    # Vcmax / Jmax / Rd (peaked Arrhenius), normalised to 1 at 25 °C
    T0 = PhysCon.tfrz + 25.0
    
    if params.vcmaxhd > 0:                          
        Vcmax = params.vcmax25 * arrhenius_function(tl, params.vcmaxha) \
            * inhibition_function(tl, params.vcmaxhd, params.vcmaxse,
                                  _fth25(params.vcmaxhd, params.vcmaxse))
    else:
        Vcmax = params.vcmax25 * arrhenius_function(tl, params.vcmaxha)
    
    Jmax = params.jmax25 * arrhenius_function(tl, params.jmaxha) \
        * inhibition_function(tl, params.jmaxhd, params.jmaxse,
                              _fth25(params.jmaxhd, params.jmaxse))
        
        
    # Day respiration -----------------------------------------------------------
    # peaked Arrhenius 
    Rd = params.rd25 * arrhenius_function(tl, params.rdha) \
        * inhibition_function(tl, params.rdhd, params.rdse,
                              _fth25(params.rdhd, params.rdse))

    # Electron transport --------------------------------------------------------
    # (plantecophys Jfun), then VJ = J/4
    aP = params.alpha_j * PAR
    disc = (aP + Jmax) ** 2 - 4.0 * params.alpha_j * params.theta_j * PAR * Jmax
    J = (aP + Jmax - np.sqrt(max(disc, 0.0))) / (2.0 * params.theta_j)
    VJ = J / 4.0

    # Medlyn slope (CO₂ basis) -------------------------------------------------- 
    # the H₂O/CO₂ ratio enters via GCtoGW below
    Ca = params.CO2_air
    vpduse = max(VPD, params.vpdmin)
    GSDIVA = (1.0 + params.g1_medlyn / (vpduse ** (1.0 - params.gk_medlyn))) / Ca
    g0c = params.g0_medlyn / params.GCtoGW

    # Coupled operating Ci for each limitation ----------------------------------
    CIJ, CIC = _getci(VJ, GSDIVA, PAR, Ca, g0c, Rd, Vcmax, Jmax, Km, Gstar)
    Ac = Vcmax * (CIC - Gstar) / (CIC + Km)
    Aj = VJ * (CIJ - Gstar) / (CIJ + 2.0 * Gstar)

    # Below light-compensation -------------------------------------------------- 
    # pin J-limited Ci to Ca (plantecophys)
    if Aj <= Rd + 1e-9:
        CIJ = Ca
        Aj = VJ * (CIJ - Gstar) / (CIJ + 2.0 * Gstar)
    ci = CIJ if Aj < Ac else CIC

    # Hyperbolic minimum (gross) and net assimilation ---------------------------
    Am = -_quadp(params.colim_c3, Ac + Aj, Ac * Aj)
    An = Am - Rd

    # Stomatal conductance ------------------------------------------------------
    # CO₂ basis → H₂O basis, floored at g0
    gsc = g0c + GSDIVA * An
    if gsc < g0c:
        gsc = g0c
    gs_mol = gsc * params.GCtoGW
    cs = Ca
                                                 
    return An, ci, cs, gs_mol

In [ ]:
#| export

def calculate_gs_medlyn(
    fluxes: SurEauPlantFluxes,
    params: SurEauVegetationParams,
    clim: dict,
) -> SurEauPlantFluxes:
    """Medlyn et al. (2011) stomatal conductance coupled to FvCB photosynthesis.

    Replacement for `calculate_gs_jarvis`stomatal conductance 
    
    SurEau's hydraulic regulation gamma (compute_regul_fact, 
    applied by compute_transpiration) is what turns it into the water-limited 
    gs_lim, so this routine has no knowledge of psi.

    Medlyn (Eq. 11):  gs = g0 + 1.6 (1 + g1/sqrt(D)) * An / cs
    
    Reads  fluxes.leaf_temperature [°C], fluxes.leaf_VPD [kPa], clim["PAR"]
    [µmol m⁻² s⁻¹].  Writes fluxes.gs_bound [mmol H₂O m⁻² s⁻¹], An, ci, cs.

    """
    
    # Solve ---------------------------------------------------------------------
    An, ci, cs, gs_mol = solve_coupled_medlyn_fvcb(
        fluxes.leaf_temperature, clim["PAR"], fluxes.leaf_VPD, params
    )
    
    #Convert mol to mmol (SurEau convention)
    fluxes.gs_bound = gs_mol * 1000.0               
    fluxes.An = An
    fluxes.ci = ci
    fluxes.cs = cs
    
    return fluxes